https://medium.com/@joshiprerak123/transform-your-audio-denoise-and-enhance-sound-quality-with-python-using-pedalboard-24da7c1df042

In [13]:
import os
import sys
from pathlib import Path
import glob

#print(os.getcwd())
root_path = Path(os.getcwd()).parent
#print(root_path)
data_path = os.path.join(root_path, "data", "raw")
sys.path.append(str(root_path))
sys.path.append(str(data_path))
#print(data_path)

In [16]:
from pathlib import Path
wav_files = list(Path(data_path).glob('*.wav'))
# print([str(p) for p in wav_files])

In [19]:
data = wav_files[0]
print(data)


c:\Users\USER\Documents\RS-SLEEP\easy_sleep\data\raw\20230904_SnoringSample_1.wav


In [4]:
import librosa
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import noisereduce as nr
import scipy
from scipy import signal
from pathlib import Path
from IPython.display import Audio, display
import soundfile as sf

matplotlib.rcParams['font.family'] = [
    'DejaVu Sans', 'Arial', 'sans-serif', 'Malgun Gothic']

noise reduce 라이브러리를 이용
- stationary 노이즈 제거: 배경 노이즈가 일정한 경우
- non-stationary 노이즈 제거: 변화하는 노이즈일 경우
- nr.reduce_noise: noise 샘플을 직접 제공하는 경우

In [ ]:
audio, sr = librosa.load(data, sr=16000)

In [6]:
start_minutes = 175
end_minutes = 180
sr = 16000

In [7]:
start_sample = int(start_minutes * 60 * sr)
print(f"start sampel: {start_sample}")
end_sample = int(end_minutes * 60 * sr)  
print(f"end sampel: {end_sample}")

start sampel: 168000000
end sampel: 172800000


In [8]:
print(f"세그먼트 길이: {(end_sample - start_sample)/sr/60:.1f}분") 

세그먼트 길이: 5.0분


In [9]:
# if end_sample > len(audio):
#     end_sample = len(audio)
#     segment = audio[start_sample:end_sample]
# else:
segment = audio[start_sample:end_sample]


In [10]:
display(Audio(segment, rate=sr))

<IPython.lib.display.Audio object>

- 로그변환을 하는 이유: 사람 귀는 소리 크기를 비선형적으로 인지함. 소리가 2배 커져도 2배 크게 느끼지 않는것을 말함. 
  - 비선형인 대신에, 비율에 민감하다고 할 수 있음. 
  - 로그는 곱셈-> 덧셈, 나눗셈-> 뺄샘으로 바꿔서 비율 관계를 선형적으로 표현함. 

In [8]:
# 경우1 
max_amp1 = 0.8
rms1 = 0.1            
ratio1 = max_amp1 / rms1
log_ratio1 = np.log10(ratio1)  # = np.log10(8) = 0.903
print(log_ratio1)


# 경우2
max_amp2 = 0.08
rms2 = 0.01           
ratio2 = max_amp2 / rms2  # = 8 (같은 비율!)
log_ratio2 = np.log10(ratio2)  # = np.log10(8) = 0.903 (같은 값!)
print(log_ratio2)

# 하지만 절대값 차이는 다름: 0.8 vs 0.08

# 로그 변환하면 비율이 같음. 
# 즉, 로그 변환 후에는 상대적인 관계(비율)에 집중할 수 있게 해줌.

0.9030899869919435
0.9030899869919435


dB 정의가 두 가지
- 전력 기준: 10 * log10(P1/P2) dB
- 진폭 기준: 20 * log10(A1/A2) B
- 이유: 전력 ∝ 진폭²이기 때문

In [9]:
# 원래 신호
amplitude_1 = 1.0
power_1 = amplitude_1 ** 2 # 전력 = 진폭의 제곱
print(f"신호 1: 진폭 = {amplitude_1}, 전력 = {power_1}")

# 진폭이 2배가 되면, 전력은 4배가 됩니다 (∵ 전력 = 진폭²)
amplitude_2 = 2.0
power_2 = amplitude_2**2  # 전력 = 진폭의 제곱
print(f"신호 2: 진폭 = {amplitude_2}, 전력 = {power_2}")

print("\n=== 비율 계산 ===")
amplitude_ratio = amplitude_2 / amplitude_1
power_ratio = power_2 / power_1
print(f"진폭 비율: {amplitude_ratio}")
print(f"전력 비율: {power_ratio}")

신호 1: 진폭 = 1.0, 전력 = 1.0
신호 2: 진폭 = 2.0, 전력 = 4.0

=== 비율 계산 ===
진폭 비율: 2.0
전력 비율: 4.0


In [10]:
print("\n=== dB 계산 ===")

# 진폭 기준 dB (20을 곱함)
db_from_amplitude = 20 * np.log10(amplitude_ratio)
print(f"진폭 기준 dB: 20 * log10({amplitude_ratio}) = {db_from_amplitude:.1f} dB")

# 전력 기준 dB (10을 곱함)
db_from_power = 10 * np.log10(power_ratio)
print(f"전력 기준 dB: 10 * log10({power_ratio}) = {db_from_power:.1f} dB")

print(f"\n결과가 같은가? {db_from_amplitude == db_from_power}")

# 진폭 기준: 20 * log10(2) = 20 * 0.3 = 6 dB
# 전력 기준: 10 * log10(4) = 10 * 0.6 = 6 dB
# 결과가 같아집니다!


=== dB 계산 ===
진폭 기준 dB: 20 * log10(2.0) = 6.0 dB
전력 기준 dB: 10 * log10(4.0) = 6.0 dB

결과가 같은가? True


- rms 레벨로 보아, 평균적인 신호 강도가 낮음 (조용한 신호)
- 최대 진폭값이 1.0을 넘어서 클리핑 발생 가능성 있음
  - 클리핑은 오디오 신호가 너무 커서 잘려나가는 현상 
  - 가끔 큰 소리(1.339) 발생 → 코골이, 뒤척임, 외부 소음 등

In [ ]:
# 주파수 대역별 에너지 분석

band_energies = nr.analyze_frequency_bands(audio, sr)

# 주파수 대역별 에너지 분석 시각화
nr.visualize_band_energies(band_energies)

   - Magnitude shape: (1025, 900001)
   - Frequency bins: 1025
주파수 대역별 에너지 분석
   - Low-Freq(20-300Hz): 0.5497 (magnitude), -5.2 dB
   - Speech(300-2000Hz): 0.7432 (magnitude), -2.6 dB
   - High-Freq(2000-8000Hz): 0.0246 (magnitude), -32.2 dB
   - Ultra-High(8000Hz+): 0.0000 (magnitude), -inf dB


In [13]:
def visualize_band_energies(band_energies):    

    # 주파수 대역별 에너지
    bands = list(band_energies.keys())
    energies = list(band_energies.values())

    # 색상을 동적으로 생성 (밴드 개수 맞춰서)
    colors = ['blue', 'orange', 'red', 'purple', 'green', 'brown', 'pink', 'gray']
    bar_colors = colors[:len(bands)]

    bars = plt.bar(range(len(bands)), energies, color=bar_colors)
    plt.title('Band Energy Distribution')
    plt.xticks(range(len(bands)), bands, rotation=45, ha='right')
    plt.ylabel('Energy')

    # 막대 위에 값 표시
    for bar, energy in zip(bars, energies):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{energy:.3f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

In [14]:
visualize_band_energies(band_energies)

<Figure size 640x480 with 1 Axes>

주파수 대역별 특성

🔵 Low-Freq (20-300Hz): 0.5655 magnitude (-5.0 dB)

가장 높은 에너지 대역
의미: 환경 소음, 에어컨, 팬 소리, 깊은 호흡음
수면 중 지속적인 배경음

🔵 Speech (300-2000Hz): 0.8054 magnitude (-1.9 dB)

최고 에너지 대역!
의미: 코골이, 중얼거림, 뒤척임 소리
수면 중 주요 활동음

🔵 High-Freq (2000-8000Hz): 0.0275 magnitude (-31.2 dB)

매우 낮은 에너지 (-30dB 이상 감소)
의미: 급격한 소음, 갑작스러운 움직임이 적음
조용한 수면 상태

🔵 Ultra-High (8000Hz+): 0.0000 magnitude (-∞ dB)

🔵 말소리 영역 (2000-4000Hz)

## 소음 제거1

- 200~1100 까지를 보통 코골이의 주파수라 생각하므로, 이를 제외한 주파수 제거

In [6]:
def noise_reduction_highfreq(audio_input, sr=16000):
    """
    연구 기반 2단계 소음 제거
    1단계: 2000Hz 이상 고주파 제거 (말소리 제거)
    2단계: noisereduce로 배경 소음 제거
    """

    if isinstance(audio_input, str):
        # 파일 경로이면
        y, sr = librosa.load(audio_input, sr=sr)
    elif isinstance(audio_input, np.ndarray):
        # 이미 로드된 배열이면
        y = audio_input  # ← y로 통일
    else:
        raise TypeError(f"Invalid audio_input type: {type(audio_input)}")

    # 1단계: 2000Hz 이상 고주파 제거 (말소리 제거)
    print("1단계: 고주파 말소리 성분 제거 중...")
    nyquist = sr / 2
    cutoff_freq = 2000  # 연구에서 입증된 cutoff 주파수
    normalized_cutoff = cutoff_freq / nyquist

    # 안전성 체크
    if normalized_cutoff >= 1.0:
        print(f"경고: 컷오프 주파수({cutoff_freq}Hz)가 나이퀴스트 주파수({nyquist}Hz)보다 높습니다!")
        normalized_cutoff = 0.99

    # Butterworth low-pass filter 설계
    b, a = signal.butter(N=5, Wn=normalized_cutoff, btype='low')

    # 필터 적용 (zero-phase filtering으로 위상 왜곡 방지)
    stage1_audio = signal.filtfilt(b, a, y)

    # 필터링 효과 확인
    original_energy = np.sum(y**2)
    filtered_energy = np.sum(stage1_audio**2)
    reduction_ratio = (1 - filtered_energy/original_energy) * 100

    print(f"완료: {cutoff_freq}Hz 이상 주파수 제거됨")
    print(f"에너지 감소: {reduction_ratio:.1f}%")

    return stage1_audio

In [7]:
reduced1 = noise_reduction_highfreq(data_path, sr=16000)

1단계: 고주파 말소리 성분 제거 중...
완료: 2000Hz 이상 주파수 제거됨
에너지 감소: 10.9%


In [ ]:
band_energies_after_reduction1 = nr.analyze_frequency_bands(reduced1, sr=16000)

   - Magnitude shape: (1025, 937501)
   - Frequency bins: 1025
주파수 대역별 에너지 분석
   - Low-Freq(20-300Hz): 0.5531 (magnitude), -5.1 dB
   - Speech(300-2000Hz): 0.7078 (magnitude), -3.0 dB
   - High-Freq(2000-8000Hz): 0.0027 (magnitude), -51.3 dB
   - Ultra-High(8000Hz+): 0.0000 (magnitude), -inf dB


In [18]:
visualize_band_energies(band_energies_after_reduction1)

<Figure size 640x480 with 1 Axes>

## 소음 제거 (환경소음 제거)

In [8]:
def noise_reduction2(audio_input, sr=16000, profile_option=1, **kwargs):
    """
    연구 기반 2단계 소음 제거
    2단계: noisereduce로 환경 소음 제거 (에어컨, 팬 등 지속적인 배경 소음)

    Parameters:
    - audio_input: 오디오 경로 또는 오디오 데이터
    - sr: 샘플링 레이트
    - profile_option: 노이즈 프로파일 선택 (1, 2, 3)
    - **kwargs: 추가 옵션들
        - noise_start_minutes: 옵션1용 노이즈 프로파일 시작 시간 (기본값: 10)
        - noise_end_minutes: 옵션1용 노이즈 프로파일종료 시간 (기본값: 20)
        - prop_decrease: 소음 제거 강도 (기본값: 0.5)
        - initial_minutes: 옵션2용 노이즈 프로파일 초기 시간 (기본값: 10)
        - final_minutes: 옵션3용 노이즈 프로파일 후반 시간 (기본값: 10)

    """
    print("=== 소음 제거 시작 ===")

    if isinstance(audio_input, str):
        print("오디오 파일 로딩 중...")
        y, sr = librosa.load(audio_input, sr=sr)
    elif isinstance(audio_input, np.ndarray):
        print("오디오 데이터 처리 중...")
        y = audio_input
    else:
        raise TypeError(f"Invalid audio_input type: {type(audio_input)}")

    print(f"오디오 길이: {len(y)/sr/3600:.1f}시간")

    # kwargs에서 파라미터 추출
    noise_start_minutes = kwargs.get("noise_start_minutes", 10)
    noise_end_minutes = kwargs.get("noise_end_minutes", 20)
    prop_decrease = kwargs.get('prop_decrease', 0.5)
    initial_minutes = kwargs.get('initial_minutes', 10)
    final_minutes = kwargs.get('final_minutes', 10)

    print("\n노이즈 프로파일 추출 중...")
    if profile_option == 1:
        noise_start_sample = int(noise_start_minutes * 60 * sr)
        noise_end_sample = int(noise_end_minutes * 60 * sr)

        if noise_end_sample >= len(y):
            noise_end_sample = len(y) - 1
            print(f"경고: 노이즈 프로파일 구간이 오디오 길이를 초과하여 조정됨")

        noise_sample = y[noise_start_sample:noise_end_sample]
        print(f"노이즈 프로파일 길이: {len(noise_sample)/sr/60:.1f}분")
        print(f"사용 구간: {noise_start_minutes}분 ~ {noise_end_minutes}분")

    elif profile_option == 2:
        print(f"초기 {initial_minutes}분을 노이즈 프로파일로 사용")
        noise_sample = y[:initial_minutes*60*sr]
        print(f"노이즈 프로파일 길이: {len(noise_sample)/sr/60:.1f}분")

    elif profile_option == 3:
        print(f"마지막 {final_minutes}분을 노이즈 프로파일로 사용")
        noise_sample = y[-final_minutes*60*sr:]
        print(f"노이즈 프로파일 길이: {len(noise_sample)/sr/60:.1f}분")

    else:
        raise ValueError(
            f"Invalid profile_option: {profile_option}. Must be 1, 2, or 3")

    print(f"\n소음 제거 적용 중... (제거 강도: {prop_decrease*100:.0f}%)")
    print("이 작업은 몇 분 정도 걸릴 수 있습니다...")

    noise_reduced_audio = nr.reduce_noise(
        y=y,
        sr=sr,
        y_noise=noise_sample,
        stationary=False,
        prop_decrease=prop_decrease
    )

    # 에너지 감소율 계산
    original_energy = np.sum(y**2)
    reduced_energy = np.sum(noise_reduced_audio**2)
    reduction_ratio = (1 - reduced_energy/original_energy) * 100

    print(f"\n소음 제거 완료!")
    print(f"전체 에너지 {reduction_ratio:.1f}% 감소")

    return noise_reduced_audio

In [9]:
reduced2 = noise_reduction2(reduced1, profile_option=1,
                 noise_start_minutes=140, noise_end_minutes=150)

=== 소음 제거 시작 ===
오디오 데이터 처리 중...
오디오 길이: 8.3시간

노이즈 프로파일 추출 중...
노이즈 프로파일 길이: 10.0분
사용 구간: 140분 ~ 150분

소음 제거 적용 중... (제거 강도: 50%)
이 작업은 몇 분 정도 걸릴 수 있습니다...

소음 제거 완료!
전체 에너지 67.7% 감소


In [12]:
def quick_waveform_comparison(original_audio, noise_reduced_audio, sr=16000, duration_minutes=60):
    """
    노이저 제거 전후 비교 시각화

    Parameters:
    - original_audio: 원본 오디오 데이터
    - noise_reduced_audio: 노이저 제거 후 오디오 데이터
    - sr: 샘플링 레이트
    - duration_minutes: 비교 시간 길이 (분)

    Returns:
    - 노이즈 제거 전후 비교 시각화
    """

    # 시각화할 구간 제한 (성능 문제로 인해)
    max_samples = int(duration_minutes * sr * 60)  # mins
    if len(original_audio) > max_samples:
        original_seg = original_audio[:max_samples]
        clean_seg = noise_reduced_audio[:max_samples]
    else:
        original_seg = original_audio
        clean_seg = noise_reduced_audio

    # 시간 축 생성
    time_axis = np.linspace(0, len(original_seg)/sr/60,
                            len(original_seg))  # 분 단위

    # 그래프 생성
    plt.figure(figsize=(16, 10))

    # 파형 비교
    plt.subplot(2, 1, 1)
    plt.plot(time_axis, original_seg, alpha=0.7,
             color='red', linewidth=1.5, label='Original')
    plt.title('Waveform of Original Audio', fontsize=14, fontweight='bold')
    plt.xlabel('Time (minutes)', fontsize=12)
    plt.ylabel('Amplitude', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.ylim([-0.25, 0.25])

    # 노이즈 제거 후 파형 비교
    plt.subplot(2, 1, 2)
    plt.plot(time_axis, clean_seg, alpha=0.7, color='blue',
             linewidth=1.5, label='Noise Reduced')
    plt.title('Waveform of Noise Reduced Audio',
              fontsize=14, fontweight='bold')
    plt.xlabel('Time (minutes)', fontsize=12)
    plt.ylabel('Amplitude', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.ylim([-0.25, 0.25])

    plt.show()

In [ ]:
quick_waveform_comparison(
    audio, reduced2, sr=16000, duration_minutes=60)

<Figure size 1600x1000 with 2 Axes>

- 위의 50분 직전의 소리는 들이마시고 내쉬는 소리가 격해지다가, 콜록거리는 소리

In [10]:
from IPython.display import Audio, display

In [11]:
# 데이터 타입과 형태 확인
print("Original audio type:", type(reduced1))
print("Original audio shape:", reduced1.shape if hasattr(
    reduced1, 'shape') else len(reduced1))
print("Reduced audio type:", type(reduced2))
print("Reduced audio shape:", reduced2.shape if hasattr(
    reduced2, 'shape') else len(reduced2))

Original audio type: <class 'numpy.ndarray'>
Original audio shape: (480000000,)
Reduced audio type: <class 'numpy.memmap'>
Reduced audio shape: (480000000,)


In [12]:
# 작은 테스트 데이터로 먼저 시도
import numpy as np

# 1초짜리 테스트 사인파
test_audio = np.sin(2 * np.pi * 440 * np.linspace(0, 1, 16000))
display(Audio(data=test_audio, rate=16000))

<IPython.lib.display.Audio object>

In [13]:
def compare_audio(original_audio, reduced_audio, sr=16000):
    """
    원본 오디오와 노이즈 제거된 오디오를 비교해서 들을 수 있게 해주는 함수
    
    Parameters:
    - original_audio: 원본 오디오 데이터
    - reduced_audio: 노이즈 제거된 오디오 데이터
    - sr: 샘플링 레이트
    """
    print("원본 오디오:")
    display(Audio(data=original_audio, rate=sr))

    print("\n노이즈 제거 후:")
    display(Audio(data=reduced_audio, rate=sr))

In [ ]:
compare_audio(
    original_audio=reduced1,
    reduced_audio=reduced2,
    sr=16000
)

In [ ]:
# 특정 구간만 듣고 싶다면:
def play_segment(audio_data, sr=16000, start_sec=0, end_sec=10):
    """
    오디오의 특정 구간만 재생
    
    Parameters:
    - audio_data (ndarray): 오디오 데이터
    - sr: 샘플링 레이트
    - start_sec: 시작 시간(초)
    - end_sec: 끝나는 시간(초)
    """
    start_idx = int(start_sec * sr)
    end_idx = int(end_sec * sr)
    segment = audio_data[start_idx:end_idx]

    print(f"{start_sec}초 ~ {end_sec}초 구간:")
    return display(Audio(data=segment, rate=sr))

In [ ]:
# 예: 30초부터 10초간 재생
print("원본 오디오 30-40초 구간:")
display(play_segment(audio, start_sec=30, end_sec=40))

print("\n노이즈 제거 후 같은 30-40초 구간:")
display(play_segment(reduced2, start_sec=30, end_sec=10))

In [ ]:
def spectrogram_comparison(original_audio, noise_reduced_audio, sr=16000, duration_minutes=60):
    """노이즈 제거 전 후 스펙트로그램 비교
    
    Parameters:
    - original_audio: 원본 오디오 데이터
    - noise_reduced_audio: 노이즈 제거 후 오디오 데이터  
    - sr: 샘플링 레이트
    - duration_minutes: 분석할 시간 (None이면 전체)
    """

    # 시각화할 구간 제한 (성능 문제로 인해)
    max_samples = int(duration_minutes * sr * 60)  # mins

    if len(original_audio) > max_samples:
        original_seg = original_audio[:max_samples]
        clean_seg = noise_reduced_audio[:max_samples]
    else:
        original_seg = original_audio
        clean_seg = noise_reduced_audio


    # 오리지널 오디오 스펙트로그램 
    stft_original = librosa.stft(original_seg, n_fft=2048, hop_length=512)
    magnitude_original = librosa.amplitude_to_db(np.abs(stft_original), ref=np.max)

    # 노이즈 제거 후 스펙트로그램
    stft_noise_reduced = librosa.stft(clean_seg, n_fft=2048, hop_length=512)
    magnitude_noise_reduced = librosa.amplitude_to_db(
        np.abs(stft_noise_reduced), ref=np.max)

    # 시간 축 계산 (STFT 프레임 기준준)
    total_time_hours = len(original_seg)/sr/60
    max_freq_khz = sr/ 2 / 1000 # kHz 단위

    # 그래프 생성
    fig, axes = plt.subplots(2,1, figsize=(16,10))

    # 스펙트로그램은 이미지 데이터로, plot이 아니라 imshow를 사용해야 함.
    img1 = axes[0].imshow(magnitude_original, aspect='auto', origin='lower',
                extent=[0, total_time_hours, 0, max_freq_khz],
                cmap='plasma', vmin=-80, vmax=0)
    axes[0].set_title('spectrogram of Original Audio', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Time (minutes)', fontsize=12)
    axes[0].set_ylabel('Frequency (Hz)', fontsize=12)
    axes[0].grid(True, linestyle='--', alpha=0.3)
    cbar1 = plt.colorbar(img1, ax=axes[0])
    cbar1.set_label('Magnitude (dB)', fontsize=12)


    # 스펙트로그램 비교
    img2 = axes[1].imshow(magnitude_noise_reduced, aspect='auto', origin='lower',
                extent=[0, total_time_hours, 0, max_freq_khz],
                cmap='plasma', vmin=-80, vmax=0)
    axes[1].set_title('spectrogram of Noise Reduced Audio',
              fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Time (minutes)', fontsize=12)
    axes[1].set_ylabel('Frequency (Hz)', fontsize=12)
    axes[1].grid(True, linestyle='--', alpha=0.3)
    cbar2 = plt.colorbar(img2, ax=axes[1])
    cbar2.set_label('Magnitude (dB)', fontsize=12)

    plt.show()

In [ ]:
spectrogram_comparison(sleep_audio, noise_reduced1, sr=16000, duration_minutes=60)

<Figure size 1600x1000 with 4 Axes>

In [ ]:
def estimate_noise_type(band_energies):
    """
    주파수 대역 에너지를 기반으로 노이즈 유형 추정
    """

    print(f"노이즈 유형 추정 \n",
          f"estimated noise type:")

    total_energy = sum(band_energies.values())

    speech_ratio = band_energies['Speech(300-2000Hz)'] / \
        total_energy if total_energy > 0 else 0
    low_freq_ratio = band_energies['Low-Freq(20-300Hz)'] / \
        total_energy if total_energy > 0 else 0
    high_freq_ratio = band_energies['High-Freq(2000-8000Hz)'] / \
        total_energy if total_energy > 0 else 0

    if speech_ratio > 0.4:
        noise_type = "speech-containing segment"
    elif high_freq_ratio > 0.3:
        noise_type = "TV/electronic device noise"
    elif low_freq_ratio > 0.4:
        noise_type = "Environmental/Background noise"
    else:
        noise_type = "Quiet segment"

    print(f"   - estimated noise type: {noise_type}")
    print(f"   - speech ratio: {speech_ratio:.1%}")
    print(f"   - low freq ratio: {low_freq_ratio:.1%}")
    print(f"   - high freq ratio: {high_freq_ratio:.1%}")

    return noise_type, speech_ratio, low_freq_ratio, high_freq_ratio


estimate_noise_type(band_energies)

노이즈 유형 추정 
 estimated noise type:
   - estimated noise type: speech-containing segment
   - speech ratio: 56.6%
   - low freq ratio: 41.5%
   - high freq ratio: 1.9%


('speech-containing segment',
 0.5656364853424468,
 0.4151331923432988,
 0.019230265310936506)

In [ ]:
def segment_noise_analysis(audio_file, start_minutes=0, end_minutes=None, duration_minutes=None, sr=16000):
    """
    지정된 구간의 노이즈 레벨을 상세 분석
    
    Parameters:
    - audio_file: 오디오 파일 경로
    - start_minutes: 시작 시간 (분)
    - end_minutes: 종료 시간 (분) - duration_minutes와 둘 중 하나만 사용
    - duration_minutes: 지속 시간 (분) - end_minutes와 둘 중 하나만 사용
    - sr: 샘플링 레이트
    
    Example:
    # 10분부터 30분까지 분석
    segment_noise_analysis('audio.wav', start_minutes=10, end_minutes=30)
    
    # 10분부터 5분 동안 분석  
    segment_noise_analysis('audio.wav', start_minutes=10, duration_minutes=5)
    """

    print(f"🔍 구간별 노이즈 분석 시작")
    print(f"파일: {audio_file}")

    # 오디오 로드 (전체 파일 메타데이터 먼저 확인)
    y_full, sr = librosa.load(audio_file, sr=sr)
    total_duration_minutes = len(y_full) / sr / 60

    print(
        f"전체 파일 길이: {total_duration_minutes:.1f}분 ({len(y_full)/sr/3600:.1f}시간)")

    # 구간 계산
    start_sample = int(start_minutes * 60 * sr)

    if end_minutes is not None:
        end_sample = int(end_minutes * 60 * sr)
        actual_duration = end_minutes - start_minutes
    elif duration_minutes is not None:
        end_sample = start_sample + int(duration_minutes * 60 * sr)
        actual_duration = duration_minutes
    else:
        # 끝까지 분석
        end_sample = len(y_full)
        actual_duration = (len(y_full) - start_sample) / sr / 60

    # 범위 체크
    end_sample = min(end_sample, len(y_full))

    print(
        f"분석 구간: {start_minutes:.1f}분 ~ {start_minutes + actual_duration:.1f}분 (총 {actual_duration:.1f}분)")

    # 해당 구간만 추출
    y_segment = y_full[start_sample:end_sample]
    segment_duration = len(y_segment) / sr

    if len(y_segment) == 0:
        print("오류: 지정된 구간이 비어있거나 범위를 벗어났습니다.")
        return None

    print(f"추출된 구간 길이: {segment_duration/60:.1f}분")

    band_energies = nr.analyze_frequency_bands(y_segment, sr=16000)

    return estimate_noise_type(band_energies)

# segment_noise_analysis(data_path, start_minutes=20, end_minutes=40, sr=16000)

In [ ]:
# 방법 2: 말소리 없는 조용한 구간을 소음 프로파일로 사용


def find_quiet_noise_profile(audio_file, sr=22050):
    """
    에너지가 낮은 조용한 구간을 소음 프로파일로 찾기
    """
    y, sr = librosa.load(audio_file, sr=sr)

    # 5분 청크로 나누어 에너지 계산
    chunk_duration = 5 * 60  # 5분
    chunk_size = int(chunk_duration * sr)

    chunk_energies = []
    for i in range(0, min(len(y), int(60*60*sr)), chunk_size):  # 첫 1시간만 확인
        chunk = y[i:i+chunk_size]
        energy = np.mean(chunk**2)
        chunk_energies.append((i, energy))

    # 가장 조용한 청크 찾기
    quietest_start = min(chunk_energies, key=lambda x: x[1])[0]
    noise_sample = y[quietest_start:quietest_start + chunk_size]

    print(
        f"가장 조용한 구간: {quietest_start/sr/60:.1f}분 ~ {(quietest_start+chunk_size)/sr/60:.1f}분")

    return noise_sample


# 테스트 코드
print("어떤 방법을 사용할지 선택하세요:")
print("1. 처음 1시간 건너뛰기")
print("2. 조용한 구간을 소음 프로파일로 사용")
print("3. 기존 방법 (처음 20분 전체)")

In [ ]:
# 2단계 소음 제거: 연구 기반 접근법

import librosa
import numpy as np
import noisereduce as nr
from scipy import signal
import matplotlib.pyplot as plt

# 연구에서 검증된 주파수 분석
def analyze_frequency_content(audio, sr):
    """
    코골이 vs 말소리 주파수 분석
    """
    # FFT 계산
    fft = np.fft.fft(audio)
    freqs = np.fft.fftfreq(len(fft), 1/sr)
    magnitude = np.abs(fft)
    
    # 양의 주파수만
    positive_freqs = freqs[:len(freqs)//2]
    positive_magnitude = magnitude[:len(magnitude)//2]
    
    # 주파수 대역별 에너지 계산
    low_freq_mask = (positive_freqs >= 40) & (positive_freqs <= 300)    # B1: 저주파
    mid_freq_mask = (positive_freqs >= 301) & (positive_freqs <= 850)   # B2: 중주파  
    high_freq_mask = (positive_freqs >= 851) & (positive_freqs <= 2000) # B3: 고주파
    speech_freq_mask = (positive_freqs >= 2000) & (positive_freqs <= 4000) # 말소리 영역
    
    low_energy = np.sum(positive_magnitude[low_freq_mask]**2)
    mid_energy = np.sum(positive_magnitude[mid_freq_mask]**2)
    high_energy = np.sum(positive_magnitude[high_freq_mask]**2)
    speech_energy = np.sum(positive_magnitude[speech_freq_mask]**2)
    
    total_energy = low_energy + mid_energy + high_energy + speech_energy
    
    print("=== 주파수 대역별 에너지 분석 ===")
    print(f"저주파 (40-300Hz): {low_energy/total_energy*100:.1f}% - 주로 코골이")
    print(f"중주파 (301-850Hz): {mid_energy/total_energy*100:.1f}% - 코골이+말소리")  
    print(f"고주파 (851-2000Hz): {high_energy/total_energy*100:.1f}% - 주로 말소리")
    print(f"말소리 영역 (2000-4000Hz): {speech_energy/total_energy*100:.1f}% - 제거 대상")
    
    return {
        'low': low_energy/total_energy,
        'mid': mid_energy/total_energy, 
        'high': high_energy/total_energy,
        'speech': speech_energy/total_energy
    }

print("2단계 소음 제거 방법이 준비되었습니다!")
print("사용법:")
print("1. two_stage_noise_reduction(audio_file) - 2단계 소음 제거")
print("2. compare_noise_reduction_methods(audio_file) - 방법 비교")
print("3. analyze_frequency_content(audio, sr) - 주파수 분석")


In [ ]:
import noisereduce as nr


def adaptive_noise_reduction(audio_file, sr=22050, chunk_duration=5, noise_duration_minutes=20):
    """
    구간별로 소음 프로파일을 적응적으로 적용
    처음 20분을 소음 프로파일로 사용
    """
    # 오디오 로드
    y, sr = librosa.load(audio_file, sr=sr)
    print(f"전체 오디오 길이: {len(y)/sr/3600:.1f}시간")

    # 처음 20분을 소음 샘플로 추출
    noise_duration_seconds = noise_duration_minutes * 60  # 20분 = 1200초
    noise_end_sample = int(noise_duration_seconds * sr)
    noise_end_sample = min(noise_end_sample, len(y))  # 파일 길이 초과 방지

    noise_sample = y[:noise_end_sample]
    print(
        f"소음 프로파일: 처음 {noise_duration_minutes}분 ({len(noise_sample)/sr/60:.1f}분) 사용")

    # 청크 크기 계산
    chunk_size = int(chunk_duration * sr)
    processed_chunks = []

    print(f"청크별 처리 시작... (청크 크기: {chunk_duration}초)")

    for i in range(0, len(y), chunk_size):
        chunk = y[i:i + chunk_size]

        # 처음 20분의 소음 프로파일을 모든 청크에 적용
        reduced_chunk = nr.reduce_noise(
            y=chunk,
            sr=sr,
            y_noise=noise_sample,  # 처음 20분을 소음 프로파일로 사용
            stationary=False,  # 비정상 소음 처리
            prop_decrease=0.8  # 소음 감소 비율 (80% 감소)
        )

        processed_chunks.append(reduced_chunk)

        # 진행상황 출력
        progress = (i + chunk_size) / len(y) * 100
        if int(progress) % 10 == 0:
            print(f"진행률: {progress:.0f}%")

    return np.concatenate(processed_chunks), sr

In [ ]:

# 소음 제거 실행
cleaned_audio, sr = adaptive_noise_reduction(
    data_path,
    sr=22050,
    chunk_duration=10,  # 10초 청크로 처리
    noise_duration_minutes=20  # 처음 20분을 소음 프로파일로 사용
)

print(f"처리 완료! 정제된 오디오 길이: {len(cleaned_audio)/sr/3600:.1f}시간")

In [ ]:
import numpy as np

ft = librosa.stft(audio) #일반 STFT 스펙트로그램 (선형 주파수) 

db = librosa.amplitude_to_db(ft, ref=np.max)  # 크기(magnitude) → dB

# waveshow는 파형용, specshow는 스펙트럼용

# y_axis는 주파수 대역, 코골이는 주로 저주파 대역 (20-1000Hz)에서 발생
# y_axis 옵션들: hz(헤르츠/주파수), mel(멜 필터뱅크/멜 스케일), log(로그 주파수/로그 스케일) 
# hz는 선형 스케일이라 저주파 영역을 자세히 볼 수 있음음
librosa.display.specshow(db, x_axis='time', y_axis='hz')
plt.show()

C:\Users\USER\AppData\Local\Temp\ipykernel_5460\2797372549.py:5: UserWarning: amplitude_to_db was called on complex input so phase information will be discarded. To suppress this warning, call amplitude_to_db(np.abs(S)) instead.
  db = librosa.amplitude_to_db(ft, ref=np.max)


<Figure size 640x480 with 1 Axes>

- librosa.feature.melspectrogram 함수는 오디오의 멜 스펙트로그램을 계산. 소리를 시간-주파수 형식으로 나타내는 것.
- librosa.power_to_db: 파워를 dB 스케일 스펙트로그램으로 변환
- librosa.display.specshow: y축에 '멜' 스케일과 x축에 시간을 사용. 

In [ ]:
import numpy as np

# 스펙트로그램 시각화
S = librosa.feature.melspectrogram(y=audio, sr=sr)
S_DB = librosa.power_to_db(S, ref=np.max)
plt.figure(figsize=(12, 4))
librosa.display.specshow(S_DB, sr=sr, x_axis='time', y_axis='mel')
plt.colorbar(format='%+2.0f dB')
plt.title('Mel Spectrogram')
plt.show()

<Figure size 1200x400 with 2 Axes>